# Demo — Onde o vetor mora: Azure Managed Redis

**Objetivo:** guardar e buscar os vetores do acervo no **Azure Managed Redis**, com o módulo RediSearch, e ler essa escolha pela pergunta do slide 21: o dado muda em tempo real? Ilustra os slides 20 a 23 (onde o vetor mora, a pergunta que decide, as opções no Azure e o algoritmo do índice).

1. **Preparar os vetores** → os mesmos trechos e o mesmo embedding dos outros notebooks;
2. **Criar a estrutura vetorial** → um índice RediSearch sobre hashes, com um campo `VECTOR` HNSW;
3. **Gravar os trechos** → um `HSET` por trecho, pela chave do trecho, sem duplicar quando o notebook roda de novo;
4. **Buscar por similaridade** → KNN no Redis: a posição do trecho certo, a similaridade e a latência da consulta;
5. **RAG completo** → a resposta com citação, a partir dos trechos que o Redis devolveu.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** um **Azure Managed Redis** num tier em memória (Memory Optimized, Balanced ou Compute Optimized; o Flash Optimized não tem RediSearch), criado com o módulo **RediSearch**. O RediSearch exige a política de cluster **Enterprise** e a política de remoção **NoEviction**. Módulo e política de cluster são escolhidos na criação, na aba **Advanced**, e o módulo não pode ser acrescentado depois. A conexão usa a porta **10000**, com TLS. A **autenticação por chave vem desligada** (o padrão é o Microsoft Entra ID): para este notebook, ligue **Access Keys Authentication** em **Settings → Authentication → Access keys**, a mesma aba que mostra as chaves.

## 0. Preparação

Este notebook usa a **Azure OpenAI v1 API** (cliente Python `OpenAI`) para os embeddings e a resposta, e o cliente `redis` (redis-py 8) para o Redis.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os mesmos valores dos outros notebooks
- `AZURE_REDIS_HOST`, no formato `<cache>.<regiao>.redis.azure.net` (está na página **Overview** do recurso)
- `AZURE_REDIS_PORT`: `10000`, a porta de todo Azure Managed Redis
- `AZURE_REDIS_KEY`: a chave primária ou a secundária, em **Settings → Authentication → Access keys**

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook. Em produção, prefira o Microsoft Entra ID com identidade gerenciada: no redis-py, isso se faz com o pacote `redis-entraid`, como no quickstart de Python do Azure Managed Redis.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "numpy>=1.26" "redis>=8.1"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
# Cliente do Redis: TLS ligado e decode_responses=False, porque o vetor é binário e não pode ser lido como texto.
import numpy as np
import redis

AZURE_REDIS_HOST = os.getenv("AZURE_REDIS_HOST") or input(
    "AZURE_REDIS_HOST (ex.: meu-cache.brazilsouth.redis.azure.net): "
).strip()
AZURE_REDIS_PORT = os.getenv("AZURE_REDIS_PORT") or "10000"
AZURE_REDIS_KEY = os.getenv("AZURE_REDIS_KEY") or getpass("AZURE_REDIS_KEY (chave de acesso): ")

NOME_INDICE = "idx:cap05"
PREFIXO = "cap05:trecho:"  # cada trecho é o hash cap05:trecho:<id do trecho>

r = redis.Redis(
    host=AZURE_REDIS_HOST,
    port=int(AZURE_REDIS_PORT),
    password=AZURE_REDIS_KEY,
    ssl=True,
    decode_responses=False,
)

try:
    r.ping()
except redis.exceptions.AuthenticationError as erro:
    raise RuntimeError(
        "O Redis recusou a chave. Confira AZURE_REDIS_KEY e se 'Access Keys Authentication' está ligada "
        "(Settings > Authentication > Access keys): no Azure Managed Redis ela vem desligada."
    ) from erro

print("Redis:", AZURE_REDIS_HOST, f"(porta {AZURE_REDIS_PORT}, TLS)")
print("Índice deste notebook:", NOME_INDICE, "| chaves:", PREFIXO + "<id do trecho>")

### Funções auxiliares

- `carregar_trechos`, `texto_para_embedding` e `gerar_embeddings`: as mesmas dos outros notebooks do Cap. 5 que as usam. Os trechos e os vetores são os que foram para o AI Search no notebook `tipos-de-rag/00`; aqui só muda onde eles moram.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import json
import re

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Um trecho por seção `## ` de cada documento, vetorizado com o deployment e a dimensão do `.env`: exatamente o que o notebook `tipos-de-rag/00-indice-compartilhado` mandou para o AI Search. Neste notebook, o que muda é **onde o vetor mora**, não como ele é calculado.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])

print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")
print(f"Em FLOAT32 (4 bytes por número), os vetores somam {len(vetores) * len(vetores[0]) * 4 / 1024:.0f} KB")

---

# Execução 2 — Criar a estrutura vetorial

No Redis, os dados ficam em chaves, e o índice do RediSearch acompanha as chaves que começam com um prefixo. Cada trecho vai ser um **hash** `cap05:trecho:<id do trecho>`, e o índice `idx:cap05` declara:

- **Texto** (`titulo`, `secao`, `conteudo`) como `TEXT`, com `LANGUAGE portuguese` para o stemming em português (o padrão do Redis é inglês);
- **Origem** (`fonte`) como `TAG`, para filtro exato;
- **Vetor** (`embedding`) como `VECTOR HNSW`, com `TYPE FLOAT32`, `DIM` igual a `EMBEDDING_DIMENSIONS` e `DISTANCE_METRIC COSINE`.

O índice só é criado se ainda não existir: `FT._LIST` lista os índices do banco, e um segundo `FT.CREATE` com o mesmo nome daria erro (`Index already exists`).

> Se aparecer `unknown command 'FT._LIST'`, o cache foi criado sem o módulo RediSearch. O módulo não pode ser acrescentado depois: é preciso criar outro cache.

In [ ]:
from redis.commands.search.field import TagField, TextField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType

campos = [
    TextField("titulo"),
    TextField("secao"),
    TextField("conteudo"),
    TagField("fonte"),
    VectorField(
        "embedding",
        "HNSW",
        {"TYPE": "FLOAT32", "DIM": EMBEDDING_DIMENSIONS, "DISTANCE_METRIC": "COSINE"},
    ),
]
# O índice acompanha as chaves com o prefixo; LANGUAGE ajusta o stemming dos campos TEXT.
definicao = IndexDefinition(prefix=[PREFIXO], index_type=IndexType.HASH, language="portuguese")

comando = ["FT.CREATE", NOME_INDICE, *definicao.args, "SCHEMA", *[a for c in campos for a in c.redis_args()]]
print("O comando que o redis-py monta:")
print(" ".join(str(a) for a in comando))

if NOME_INDICE.encode() in r.execute_command("FT._LIST"):
    print(f"\nO índice '{NOME_INDICE}' já existe: nada a criar.")
else:
    r.ft(NOME_INDICE).create_index(campos, definition=definicao)
    print(f"\nÍndice '{NOME_INDICE}' criado: HNSW, cosseno, {EMBEDDING_DIMENSIONS} dimensões em FLOAT32.")

### Os limites desta base (slide 22)

- A linha do Redis: **até 32.768 dimensões · índices HNSW e Flat · reranking: não**. Os 1.536 números do `text-embedding-3-small` cabem com folga: aqui, o limite de dimensão não decide nada.
- **A memória é o orçamento.** Só os vetores de 1 milhão de trechos de 1.536 dimensões, em FLOAT32, somam cerca de 6,1 GB, antes do grafo HNSW e do texto; e o serviço reserva cerca de 20% da memória do cache. Com o RediSearch a política é `NoEviction`: memória cheia não apaga chave, faz a escrita falhar.

### O algoritmo do índice (slide 23)

- O **HNSW** é o grafo em camadas, em memória: recall alto e lida bem com dado que muda, ao custo de memória.
- A outra opção do Redis, o **FLAT**, é a busca exata (força bruta), que a documentação do Redis indica para conjuntos pequenos, abaixo de 1 milhão de vetores. Com 50 trechos, o FLAT bastaria, e os dois tendem a devolver o mesmo resultado. Usamos HNSW para mostrar o algoritmo do slide 23, o que a documentação do Redis indica quando o acervo passa de 1 milhão de vetores. Trocar é mudar uma palavra no `FT.CREATE` e recriar o índice.

---

# Execução 3 — Gravar os trechos

Cada trecho vira um `HSET` na chave `cap05:trecho:<id do trecho>`, com o texto, a origem e o vetor em bytes. Como a chave é o `id` estável do trecho (documento + número da seção), rodar o notebook de novo **sobrescreve** os mesmos hashes, em vez de duplicar. O `HSET` devolve quantos campos novos criou: 5 num trecho novo, 0 num trecho regravado.

Os 50 `HSET` vão num pipeline **sem transação** (`transaction=False`): o pipeline só junta os comandos numa ida e volta pela rede. Com a política de cluster Enterprise, uma transação (`MULTI`) com chaves de slots diferentes pode falhar com `CROSSSLOT`.

Depois da escrita, o `FT.INFO` mostra quantos trechos o índice enxerga (`num_docs`). A leitura vem logo depois dos `HSET`, sem a espera que o AI Search pede (`tipos-de-rag/00` e `01-ai-search` desta série): a contagem já deve mostrar os 50 aqui; confira o número na saída da célula. Se vier menos, algum trecho ficou fora do índice, e a causa provável é um vetor com dimensão diferente do `DIM`: o `HSET` grava o hash sem dar erro, mas a indexação falha, e o trecho fica fora da busca. Por isso o aviso da célula olha para o `num_docs`, e não para o `hash_indexing_failures`: esse contador acumula as falhas desde a criação do índice e não diminui quando o trecho é corrigido.

O `vector_index_sz_mb` é a memória do índice vetorial, e não só a dos vetores crus da Execução 1: o índice guarda a própria cópia de cada vetor (a outra fica no hash), o grafo do HNSW e uma reserva de memória feita em blocos, antes de eles se encherem. Por isso, mesmo com 50 trechos, o número pode passar bem do total da Execução 1: no Redis, a memória é o orçamento.

In [ ]:
pipe = r.pipeline(transaction=False)
for trecho, vetor in zip(trechos, vetores):
    pipe.hset(
        PREFIXO + trecho["id"],
        mapping={
            "titulo": trecho["titulo"],
            "secao": trecho["secao"],
            "conteudo": trecho["conteudo"],
            "fonte": trecho["fonte"],
            "embedding": np.array(vetor, dtype=np.float32).tobytes(),
        },
    )
campos_novos = pipe.execute()  # um número por HSET: quantos campos novos ele criou
novos = sum(n > 0 for n in campos_novos)
print(f"HSET enviados: {len(campos_novos)} | trechos novos: {novos} | regravados: {len(campos_novos) - novos}")

info = r.ft(NOME_INDICE).info()
indexados = int(info["num_docs"])
falhas = int(info["hash_indexing_failures"])  # contador acumulado desde o FT.CREATE: não diminui
memoria_mb = float(info.get("vector_index_sz_mb", "nan"))
print(f"Documentos no índice: {indexados} (trechos no acervo: {len(trechos)})")
print(f"Falhas de indexação acumuladas desde a criação do índice: {falhas}")
print(f"Memória do índice vetorial, segundo o FT.INFO: {memoria_mb:.2f} MB")
if indexados < len(trechos):
    print("Há trechos fora do índice: provavelmente o vetor não tem a dimensão do índice (DIM). O HSET grava o hash")
    print("mesmo assim, mas o trecho fica fora da busca. Rode a limpeza com APAGAR_RECURSOS = True e depois o notebook inteiro.")
elif falhas:
    print("Todos os trechos estão no índice: as falhas acima vêm de gravações anteriores (o contador não diminui).")

---

# Execução 4 — Buscar por similaridade

A pergunta vira vetor **no cliente**, com o mesmo `gerar_embeddings`, e o Redis devolve os `k` vizinhos mais próximos com uma consulta KNN: `*=>[KNN 5 @embedding $vetor AS distancia]`, com `SORTBY distancia` e `DIALECT 2` (a busca vetorial exige o dialeto 2 ou superior).

O Redis devolve a **distância de cosseno**: 0 quando os vetores apontam para o mesmo lado, até 2 no oposto. Para ler como nos outros notebooks, `buscar` converte: **similaridade = 1 − distância**, que é o próprio cosseno entre os vetores (quanto maior, mais próximo). Não é probabilidade de acerto, e não é o `@search.score` do AI Search, que usa outra régua (`1 / (1 + distância)`).

Rodamos duas perguntas do gabarito: uma em que o vetor tende a acertar (linguagem natural) e uma em que ele tende a tropeçar (string exata, o SKU 48213-B).

### O que esperar e por que pode variar

- Na pergunta em linguagem natural, o trecho "Como pedir o reembolso de uma viagem" tende a vir em 1º.
- No SKU 48213-B, os três SKUs (B, C e D) tendem a aparecer juntos no topo, com similaridades muito próximas: é o quase empate do slide 14. O trecho certo pode vir em 1º por sorte, ou perder para um irmão.
- Os números dependem do modelo e da dimensão do embedding, e o HNSW é aproximado: não trate a tabela como gabarito fixo.
- A latência medida é só a da consulta ao Redis, sem o embedding da pergunta. De um notebook fora do Azure, quase todo esse tempo é a ida e volta pela rede; o KNN em memória, com 50 vetores, é uma fração dele.

In [ ]:
import time

from redis.commands.search.query import Query

LATENCIAS_MS = []  # tempo de cada consulta ao Redis, sem contar o embedding da pergunta


def buscar(pergunta: str, k: int = 5) -> list[dict]:
    """KNN no Redis: a pergunta vira vetor no cliente e o índice devolve os k trechos mais próximos."""
    vetor = np.array(gerar_embeddings([pergunta])[0], dtype=np.float32).tobytes()
    consulta = (
        Query(f"*=>[KNN {k} @embedding $vetor AS distancia]")
        .sort_by("distancia")
        .return_fields("titulo", "secao", "conteudo", "fonte", "distancia")
        .paging(0, k)
        .dialect(2)
    )
    inicio = time.perf_counter()
    resultado = r.ft(NOME_INDICE).search(consulta, query_params={"vetor": vetor})
    LATENCIAS_MS.append((time.perf_counter() - inicio) * 1000)
    # O redis-py decodifica em UTF-8 os campos pedidos em return_fields; o id do trecho é a chave sem o prefixo.
    return [
        {
            "id": doc.id.removeprefix(PREFIXO),
            "titulo": doc.titulo,
            "secao": doc.secao,
            "conteudo": doc.conteudo,
            "fonte": doc.fonte,
            "similaridade": 1 - float(doc.distancia),  # distância de cosseno (0 a 2) → cosseno (1 a -1)
        }
        for doc in resultado.docs
    ]


resultados = {}
for pid in ["linguagem-natural", "string-exata"]:
    p = PERGUNTAS[pid]
    top5 = buscar(p["pergunta"], k=5)
    resultados[pid] = top5
    esperados = p["trechos_esperados"][0]
    lugar = posicao([t["id"] for t in top5], esperados)
    situacao = f"{lugar}º lugar" if lugar else "fora do top 5"
    display(Markdown(f"**{p['pergunta']}**  \nTrecho certo: {situacao} · consulta KNN no Redis: {LATENCIAS_MS[-1]:.1f} ms"))
    display(pd.DataFrame([
        {
            "posição": i,
            "trecho": f"{t['titulo']} — {t['secao']}",
            "distância": round(1 - t["similaridade"], 4),
            "similaridade": round(t["similaridade"], 4),
            "diferença para o 1º": round(t["similaridade"] - top5[0]["similaridade"], 4),
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

### O que observar

- **Trocar a base não muda a natureza do vetor.** O quase empate SKU 48213-B x C x D do slide 14 continua aqui, como no notebook `tipos-de-rag/01`: os vetores são os mesmos, só mudaram de casa. Veja a coluna **diferença para o 1º**. Lá a diferença está na escala do `@search.score`, por isso parece menor: compare a ordem, não os números.
- O remédio para o empate é decisão de **recuperação**: busca híbrida e reranking (`tipos-de-rag/02` e `03`). A escolha da base é decisão de **operação** (slide 21): quem já opera, quem monitora, quem paga.
- Confira a latência de cada consulta: é a "latência de milissegundos" da linha do Redis no slide 22. Rode a célula de novo: o número oscila mais com a rede do que com o tamanho do acervo.

---

# Execução 5 — RAG completo

Os 3 primeiros trechos que o Redis devolveu para a pergunta em linguagem natural vão para o modelo, que responde citando o `id` de cada trecho. É o mesmo `responder` das outras bases da série e de `tipos-de-rag`: a geração não sabe, nem precisa saber, onde o vetor mora.

### O que esperar e por que pode variar

- Se o trecho certo estiver entre os três enviados, a resposta tende a sair certa e a citar `[financeiro_reembolso-de-despesas-de-viagem_01]`.
- O texto muda a cada execução. O que importa é a citação apontar para um trecho que foi enviado.

In [ ]:
pid = "linguagem-natural"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados[pid][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração (slide 29).

### Leitura arquitetural

- **Quando o Redis é a escolha certa**, pelo slide 22: *"Latência de milissegundos, com conjunto que cabe em memória"*. Leia isso pela pergunta do slide 21, "o dado muda em tempo real?": o Redis faz sentido quando o vetor acompanha um dado que muda o tempo todo e já mora no Redis (sessão, catálogo, cache semântico, memória de agente), ou quando a casa já opera Redis. É o ramo que o guia de busca vetorial do Azure Architecture Center indica para busca vetorial em memória, de latência muito baixa. Para um acervo estável de documentos, como este, o slide 21 aponta o índice dedicado (AI Search).
- **Quem opera, monitora e paga.** A Microsoft opera o serviço: infraestrutura, réplicas e atualização de versão. O time continua dono do resto: dimensionar a memória (vetores, grafo HNSW e texto), vigiar o uso (com `NoEviction`, memória cheia faz a escrita falhar), regravar os vetores quando o modelo do embedding muda, recriar o índice quando a dimensão muda e pagar pelo tier e pelo tamanho de cache escolhidos.
- **Busca híbrida, no Azure Managed Redis de hoje (Redis 7.4), é vetor + filtro de atributos na mesma consulta**, como a descreve o guia do Azure Architecture Center: `(@conteudo:rolamento)=>[KNN 5 @embedding $vetor AS distancia]` só considera os trechos que falam de rolamento e os ordena pelo vetor. Não é a fusão de dois rankings (BM25 + vetor, com RRF) do AI Search: no Redis, essa fusão é o comando `FT.HYBRID`, que só existe a partir do Redis 8.4 (o serviço atualiza a versão automaticamente quando uma nova fica disponível no Azure). E não há reranking embutido (slide 22): o que faltar vira código da aplicação.

### Fala sugerida

> “Trocar de banco não desempata o SKU. O banco decide quem opera, monitora e paga; quem desempata é a recuperação.”

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos 50 trechos e vetores dos outros notebooks | O contrato de embedding (modelo e dimensão) não depende da base |
| 2 | Índice RediSearch sobre hashes, com `VECTOR HNSW`, FLOAT32, cosseno e o `DIM` do `.env` | Algoritmo e dimensão ficam no `FT.CREATE`; módulo e política de cluster, na criação do cache |
| 3 | `HSET` pela chave do trecho: regravar não duplica, e o índice acompanha a escrita | Chave = `id` estável; conferir o `num_docs` do `FT.INFO`, porque vetor com outro `DIM` fica fora do índice sem erro no `HSET` |
| 4 | KNN em milissegundos, e o quase empate do SKU continua lá | A base não muda o vetor: híbrido e reranking são decisões de recuperação |
| 5 | A resposta cita os trechos que o Redis devolveu | A geração não sabe onde o vetor mora |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Azure Managed Redis — criar o cache (módulos, política de cluster e autenticação):  
  https://learn.microsoft.com/azure/redis/quickstart-create-managed-redis

- Azure Managed Redis — vetores e busca vetorial com o RediSearch:  
  https://learn.microsoft.com/azure/redis/overview-vector-similarity

- Azure Managed Redis — Microsoft Entra ID e chaves de acesso:  
  https://learn.microsoft.com/azure/redis/entra-for-authentication

- Azure Architecture Center — escolher um serviço do Azure para busca vetorial (dimensões, híbrido e reranking):  
  https://learn.microsoft.com/azure/architecture/guide/technology-choices/vector-search

---

# Limpeza (opcional)

A célula abaixo apaga só o que este notebook criou: o índice `idx:cap05` e os 50 hashes `cap05:trecho:*`. São dois passos porque, no Redis, índice e dados são coisas separadas: o `FT.DROPINDEX` sem a opção `DD` apaga só o índice e deixa as chaves. O cache continua no Azure; se ele foi criado só para a aula, apague-o no portal.

A célula pode rodar mais de uma vez: o `FT._LIST` confere se o índice ainda existe antes do `FT.DROPINDEX`. Ela também não depende dos embeddings, porque as chaves saem dos `id` do acervo: depois de reiniciar o kernel, basta rodar a Preparação (seção 0) e esta célula.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, os vetores ficariam fora da busca, porque o `DIM` de um índice existente não muda (a Execução 3 avisa). Apague aqui e rode o notebook inteiro de novo, com `APAGAR_RECURSOS` de volta em `False`.

Mudou só o modelo, com a mesma dimensão? Nada falha: o `DIM` é o mesmo, e o `HSET` regrava os 50 vetores pela mesma chave. Reinicie o kernel e rode o notebook inteiro (slide 10: vetores de modelos diferentes não se comparam).

In [ ]:
APAGAR_RECURSOS = False  # True apaga o índice e os hashes deste notebook

if APAGAR_RECURSOS:
    if NOME_INDICE.encode() in r.execute_command("FT._LIST"):
        r.ft(NOME_INDICE).dropindex(delete_documents=False)  # só o índice; as chaves saem no DEL abaixo
        print(f"Índice '{NOME_INDICE}' apagado.")
    else:
        print(f"O índice '{NOME_INDICE}' já não existia.")
    chaves = [PREFIXO + t["id"] for t in carregar_trechos(PASTA_ACERVO)]  # só os ids: nenhum embedding
    apagadas = r.delete(*chaves)  # DEL entre slots é aceito na política Enterprise
    print(f"Hashes '{PREFIXO}...' apagados: {apagadas}.")
else:
    print(f"Nada foi apagado. O índice '{NOME_INDICE}' e os hashes '{PREFIXO}...' continuam no Redis.")